# 3.3 Linear Regression

**Class 4.** Logistic regression (3.2) models a **yes/no** outcome. When the outcome is a **number on a scale**, like blood pressure, we use **linear regression**. Here we ask: which factors are associated with **systolic blood pressure**, and how much does each one add to our ability to explain it?

### By the end of this notebook you'll be able to

1. Plot a variable's distribution and a relationship between two variables with **ggplot2**
2. Fit and interpret a **linear regression** with `lm()`: what each coefficient means
3. Compare models with **R²**, **adjusted R²**, **AIC**, and **BIC**
4. Check a model's **assumptions** with diagnostic plots

| | |
|---|---|
| ⏱️ **Time** | 45 minutes |
| 📂 **Uses** | `final_analysis.csv` |
| 💾 **Saves** | nothing |

**Before you start:**

- Run **2.0 clean_and_join**
- It helps to have done **3.2 logistic_regression**

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Key concepts

### A straight line through the data

Linear regression fits the line that best describes how the outcome changes with a predictor:

**Systolic BP = β₀ + β₁ × age**

- **β₀, the intercept:** the predicted BP when age = 0. Often not meaningful by itself (no one in our data is 0).
- **β₁, the slope (coefficient):** how much BP changes, on average, for each **1-year** increase in age, in the outcome's own units (**mm Hg**).

With several predictors, each coefficient is the change in the outcome for a 1-unit change in **that** predictor, **holding the others constant**. For a categorical predictor (like gender), it's the difference from the **reference** category.

### How well does a model fit?

| Measure | Means | Better is |
|---|---|---|
| **R²** | the share of the variation in the outcome the model explains (0 to 1) | higher |
| **Adjusted R²** | R², penalised for each extra predictor; use it to compare models of different sizes | higher |
| **AIC / BIC** | balance fit against complexity; BIC penalises extra predictors more | **lower**; a drop of more than ~2 is meaningful |

### Assumptions (LINE)

Linear regression assumes: a **L**inear relationship, **I**ndependent observations, **N**ormally distributed residuals (the differences between observed and predicted values), and **E**qual variance of residuals across predicted values. We check these with **diagnostic plots**.

In [ ]:
source("aou_setup.R")  # settings, packages, and helper functions

In [ ]:
library(broom)

fa <- prepare_analysis_data()

## Look at the outcome

Always **plot your data** before modelling. A **histogram** shows how the values are spread out, and reveals skew, outliers, or impossible values (like a BP of 0).

**ggplot2 in one minute.** Plots are built in layers, joined with `+`:

- `ggplot(data, aes(x = …, y = …))` says which data to use and which columns go on which axis (`aes` = *aesthetics*)
- `geom_…()` adds a layer: `geom_histogram()` bars, `geom_point()` dots, `geom_smooth()` a fitted line
- `labs()` sets the axis labels

> ⚠️ Plots of participant data are fine **inside** the Workbench. Don't screenshot or share them if they could reveal individuals or small groups.

In [ ]:
ggplot(fa, aes(Systolic_BP)) +
  geom_histogram(bins = 30) +
  labs(x = "Systolic blood pressure (mm Hg)", y = "Participants")

**🔍 Read the output:** most values should fall roughly between 90 and 160 mm Hg, in a hump shape. A long tail on one side means **skew**; isolated bars far from the rest are **outliers** worth checking.

## A simple model: age only

`lm()` fits a **l**inear **m**odel. `tidy(conf.int = TRUE)` returns the coefficients with 95% confidence intervals. Unlike logistic regression, we **don't** exponentiate: the coefficients are already in mm Hg.

In [ ]:
m_age <- lm(Systolic_BP ~ age, data = fa)
tidy(m_age, conf.int = TRUE)

**🔍 Read the output:** the `age` row's `estimate` is **β₁**: the average change in systolic BP per year of age.

**Interpret:** *"Each additional year of age is associated with an average change of β₁ mm Hg in systolic BP."* Over 10 years, that's 10 × β₁. Is that clinically meaningful?

**See the model.** A scatterplot with the fitted line. `alpha = 0.1` makes the dots see-through, so dense areas look darker; `geom_smooth(method = "lm")` draws the regression line with its confidence band.

In [ ]:
ggplot(fa, aes(age, Systolic_BP)) +
  geom_point(alpha = 0.1) +
  geom_smooth(method = "lm", se = TRUE) +
  labs(x = "Age (years)", y = "Systolic BP (mm Hg)")

### ✍️ Your turn

Use `glance(m_age)` (from **broom**) to see how much of the variation in blood pressure **age alone** explains. Look at `r.squared`.

<details>
<summary><b>Show answer</b></summary>

```r
glance(m_age)
```

`r.squared` is usually small here, e.g. 0.04 = 4%. Blood pressure depends on many things besides age, so one predictor explains only a little. That doesn't make age unimportant: a small R² can still come with a real, meaningful slope.

</details>

In [ ]:
# Your code here

## Multivariable models

We build up in steps, as in 3.2:

| Model | Predictors |
|---|---|
| Demog | age, gender, race, ethnicity |
| +BMI+LDL | adds BMI and LDL cholesterol |
| +Diabetes | adds diabetes status: our **flagging variable** from 2.0 (No / Yes) |

`glance()` gives one row of fit statistics per model; `bind_rows(…, .id = "model")` stacks them with the model names.

In [ ]:
m0 <- lm(Systolic_BP ~ age + gender + race + ethnicity, data = fa)
m1 <- update(m0, . ~ . + BMI + LDL_Cholesterol)
m2 <- update(m1, . ~ . + diabetes)   # the flagging variable (No / Yes)

bind_rows(Demog = glance(m0), `+BMI+LDL` = glance(m1), `+Diabetes` = glance(m2), .id = "model") %>%
  select(model, r.squared, adj.r.squared, AIC, BIC)

**🔍 Read the output:** compare across rows.

- How much does **adjusted R²** go up when we add BMI and LDL?
- Does **AIC** drop by more than about 2? Does **BIC**?
- Does adding **diabetes** improve the fit further, or barely change it?

## Discuss

1. How much does adding **BMI and LDL** improve the fit? What does that tell you about blood pressure?
2. Does adding **diabetes** improve it further? If not, should it stay in the model? (Sometimes a variable stays because it's part of the research question, even if it doesn't improve fit.)
3. In model `m2`, the `diabetesYes` coefficient is the difference in BP between people with and without diabetes, **holding the other variables constant**. Look at it with `tidy(m2, conf.int = TRUE)`.

### ✍️ Your turn

Check the assumptions of model `m1` with its **diagnostic plots**. Run `plot(m1)`: it draws four plots. What should each look like?

<details>
<summary><b>Show answer</b></summary>

```r
par(mfrow = c(2, 2))   # 2 x 2 grid of plots
plot(m1)
par(mfrow = c(1, 1))   # back to one plot at a time
```

| Plot | Good sign |
|---|---|
| **Residuals vs Fitted** | a shapeless cloud around 0 (**linearity**) |
| **Q-Q plot** | points close to the diagonal line (**normal** residuals) |
| **Scale-Location** | a roughly flat band (**equal variance**) |
| **Residuals vs Leverage** | no points far outside the dashed lines (no single person drives the results) |

With thousands of people, small departures are common and usually not serious.

</details>

In [ ]:
# Your code here

## ⚠️ Common errors

| Problem | Fix |
|---|---|
| *object 'Systolic_BP' not found* | Check spelling and capitals; run `names(fa)` |
| `geom_smooth()` message *using formula y ~ x* | Just information, not an error |
| The plot is tiny or squashed | Resize with `options(repr.plot.width = 10, repr.plot.height = 6)` |

## ✅ Key takeaways

- Linear regression models a **continuous** outcome; coefficients are in the outcome's **units**
- Each coefficient is the change in the outcome per 1-unit change in a predictor, **holding the others constant**
- Compare models with **adjusted R²**, **AIC**, and **BIC**, not R² alone
- **Plot** the data first, and check the **assumptions** after

**Next:** **3.4 survival_analysis**.